In [40]:
import pandas as pd
import numpy as np

# данные лежат в соседней папке data/raw/, поэтому ../data/
race_results = pd.read_csv("../data/raw/race_results.csv")
quali_results = pd.read_csv("../data/raw/quali_results.csv")
drivers = pd.read_csv("../data/raw/drivers.csv")
weather = pd.read_csv("../data/raw/weather.csv")
meetings = pd.read_csv("../data/raw/meetings.csv")
laps_agg = pd.read_csv("../data/raw/laps_agg.csv")
sessions = pd.read_csv("../data/raw/sessions.csv")

print("результаты гонок:", race_results.shape)
print("результаты квал:", quali_results.shape)
print("пилоты:", drivers.shape)
print("погода:", weather.shape)
print("трассы:", meetings.shape)
print("телеметрия:", laps_agg.shape)
print("сессии:", sessions.shape)

результаты гонок: (1397, 11)
результаты квал: (1377, 10)
пилоты: (1397, 5)
погода: (70, 6)
трассы: (74, 7)
телеметрия: (1394, 9)
сессии: (364, 15)


In [41]:
# колонки трассы, которые берем (используем и для гонки, и для квалы)
track_cols = ["meeting_key", "country_name", "circuit_short_name", "circuit_type", "year"]

In [42]:
race = race_results.copy()

# пилоты и команды (по сессии + номеру)
race = race.merge(drivers, on=["session_key", "driver_number"], how="left")

# телеметрия (по сессии + номеру)
race = race.merge(laps_agg, on=["session_key", "driver_number"], how="left")

# погода (по сессии)
race = race.merge(weather, on="session_key", how="left")

# трасса (по уикенду)
race = race.merge(meetings[track_cols], on="meeting_key", how="left")

print("гонка:", race.shape)

гонка: (1397, 30)


In [43]:
# команды привязываем к уикенду (meeting_key), а не к сессии квалы
driver_team = race_results[["meeting_key", "driver_number"]].merge(
    drivers[["driver_number", "team_name", "full_name", "name_acronym"]],
    on="driver_number", how="left"
)
driver_team = driver_team.drop_duplicates(subset=["meeting_key", "driver_number"])

# погоду тоже привязываем к уикенду
weather_meeting = race_results[["session_key", "meeting_key"]].drop_duplicates().merge(
    weather, on="session_key", how="left"
)
weather_cols = ["meeting_key", "air_temp_mean", "track_temp_mean", "humidity_mean", "wind_speed_mean", "rainfall_max"]
weather_meeting = weather_meeting[weather_cols].drop_duplicates(subset=["meeting_key"])

# собираем квалу
quali = quali_results.copy()
quali = quali.merge(driver_team, on=["meeting_key", "driver_number"], how="left")
quali = quali.merge(weather_meeting, on="meeting_key", how="left")
quali = quali.merge(meetings[track_cols], on="meeting_key", how="left")

print("квала:", quali.shape)
print("команды заполнены:", quali["team_name"].notna().sum(), "из", len(quali))

квала: (1377, 22)
команды заполнены: 1375 из 1377


In [44]:
session_dates = sessions[["session_key", "date_start"]].copy()
session_dates["date_start"] = pd.to_datetime(session_dates["date_start"])

race = race.merge(session_dates, on="session_key", how="left")
race = race.sort_values("date_start").reset_index(drop=True)

quali = quali.merge(session_dates, on="session_key", how="left")
quali = quali.sort_values("date_start").reset_index(drop=True)

print("гонка:", race.shape, "| даты:", race["date_start"].min(), "—", race["date_start"].max())
print("квала:", quali.shape, "| даты:", quali["date_start"].min(), "—", quali["date_start"].max())

гонка: (1397, 31) | даты: 2023-03-05 15:00:00+00:00 — 2025-12-07 13:00:00+00:00
квала: (1377, 23) | даты: 2023-03-04 15:00:00+00:00 — 2025-12-06 14:00:00+00:00


In [45]:
import os
os.makedirs("../data/raw", exist_ok=True)

race.to_csv("../data/raw/dataset_race_raw.csv", index=False)
quali.to_csv("../data/raw/dataset_quali_raw.csv", index=False)

print("сохранено:")
print("гонка:", race.shape)
print("квала:", quali.shape)

сохранено:
гонка: (1397, 31)
квала: (1377, 23)


In [46]:
import pandas as pd
import numpy as np

# данные лежат в соседней папке data/raw/, поэтому ../data/
race_results = pd.read_csv("../data/raw/race_results.csv")
quali_results = pd.read_csv("../data/raw/quali_results.csv")
drivers = pd.read_csv("../data/raw/drivers.csv")
weather = pd.read_csv("../data/raw/weather.csv")
meetings = pd.read_csv("../data/raw/meetings.csv")
laps_agg = pd.read_csv("../data/raw/laps_agg.csv")
sessions = pd.read_csv("../data/raw/sessions.csv")

print("результаты гонок:", race_results.shape)
print("результаты квал:", quali_results.shape)
print("пилоты:", drivers.shape)
print("погода:", weather.shape)
print("трассы:", meetings.shape)
print("телеметрия:", laps_agg.shape)
print("сессии:", sessions.shape)

результаты гонок: (1397, 11)
результаты квал: (1377, 10)
пилоты: (1397, 5)
погода: (70, 6)
трассы: (74, 7)
телеметрия: (1394, 9)
сессии: (364, 15)


этап 2 - feature engineering

driver_form_5 — средняя позиция пилота за последние 5 прошлых гонок (недавняя форма)
driver_form_all — средняя позиция за все прошлые гонки (общий уровень) ← с этой начали
team_form_5 — средняя позиция команды за последние 5 гонок (сила болида)
driver_track_history — средняя позиция пилота на этой трассе раньше
driver_finish_rate — доля прошлых гонок без схода (надёжность)
driver_experience — сколько гонок проехал до текущей (опыт)
driver_speed_5 — средняя макс-скорость за прошлые 5 гонок (телеметрия)

In [47]:
# сортируем по пилоту и дате: у каждого пилота гонки по порядку времени
race = race.sort_values(["driver_number", "date_start"]).reset_index(drop=True)

# шаг 1: сдвигаем позицию на гонку назад внутри пилота (прячем текущую гонку)
race["pos_shifted"] = race.groupby("driver_number")["position"].shift(1)

# шаг 2: средняя позиция за ВСЕ прошлые гонки (накопительное среднее)
race["driver_form_all"] = (
    race.groupby("driver_number")["pos_shifted"].expanding().mean().values
)

# шаг 3: средняя позиция за последние 5 прошлых гонок (недавняя форма)
race["driver_form_5"] = (
    race.groupby("driver_number")["pos_shifted"].rolling(5).mean().values
)

# убираем временную колонку
race = race.drop(columns=["pos_shifted"])

# возвращаем сортировку по дате
race = race.sort_values("date_start").reset_index(drop=True)

race[["date_start", "name_acronym", "position", "driver_form_all", "driver_form_5"]].head(15)

,date_start,name_acronym,position,driver_form_all,driver_form_5
0,2023-03-05 15:00:00+00:00,VER,1.0,NaN,NaN
1,2023-03-05 15:00:00+00:00,SAR,12.0,NaN,NaN
2,2023-03-05 15:00:00+00:00,ALB,10.0,NaN,NaN
3,2023-03-05 15:00:00+00:00,OCO,NaN,NaN,NaN
4,2023-03-05 15:00:00+00:00,TSU,11.0,NaN,NaN
5,2023-03-05 15:00:00+00:00,NOR,17.0,NaN,NaN
6,2023-03-05 15:00:00+00:00,DEV,14.0,NaN,NaN
7,2023-03-05 15:00:00+00:00,MAG,13.0,NaN,NaN
8,2023-03-05 15:00:00+00:00,HUL,15.0,NaN,NaN
9,2023-03-05 15:00:00+00:00,BOT,8.0,NaN,NaN


In [48]:
# смотрим на Ферстаппена (VER) по ходу сезона: как растет его форма
ver = race[race["name_acronym"] == "VER"]
print(ver[["date_start", "position", "driver_form_all", "driver_form_5"]].head(10))

                   date_start  position  driver_form_all  driver_form_5
0   2023-03-05 15:00:00+00:00       1.0              NaN            NaN
39  2023-03-19 17:00:00+00:00       2.0         1.000000            NaN
48  2023-04-02 05:00:00+00:00       1.0         1.500000            NaN
75  2023-04-30 11:00:00+00:00       2.0         1.333333            NaN
89  2023-05-07 19:30:00+00:00       1.0         1.500000            NaN
114 2023-05-28 13:00:00+00:00       1.0         1.400000            1.4
126 2023-06-04 13:00:00+00:00       1.0         1.333333            1.4
152 2023-06-18 18:00:00+00:00       1.0         1.285714            1.2
169 2023-07-02 13:00:00+00:00       1.0         1.250000            1.2
194 2023-07-09 14:00:00+00:00       1.0         1.222222            1.0


In [49]:
# сортируем по команде и дате
race = race.sort_values(["team_name", "date_start"]).reset_index(drop=True)

# сдвигаем позицию на гонку назад внутри команды (прячем текущую гонку)
race["pos_shifted"] = race.groupby("team_name")["position"].shift(1)

# средняя позиция команды за последние 5 прошлых гонок (сила болида)
race["team_form_5"] = race.groupby("team_name")["pos_shifted"].rolling(5).mean().values

# убираем временную колонку
race = race.drop(columns=["pos_shifted"])

# возвращаем сортировку по дате
race = race.sort_values("date_start").reset_index(drop=True)

race[["date_start", "team_name", "name_acronym", "position", "team_form_5"]].head(15)

,date_start,team_name,name_acronym,position,team_form_5
0,2023-03-05 15:00:00+00:00,Alfa Romeo,BOT,8.0,NaN
1,2023-03-05 15:00:00+00:00,Ferrari,SAI,4.0,NaN
2,2023-03-05 15:00:00+00:00,Haas F1 Team,MAG,13.0,NaN
3,2023-03-05 15:00:00+00:00,Haas F1 Team,HUL,15.0,NaN
4,2023-03-05 15:00:00+00:00,AlphaTauri,TSU,11.0,NaN
5,2023-03-05 15:00:00+00:00,AlphaTauri,DEV,14.0,NaN
6,2023-03-05 15:00:00+00:00,McLaren,PIA,NaN,NaN
7,2023-03-05 15:00:00+00:00,Williams,SAR,12.0,NaN
8,2023-03-05 15:00:00+00:00,McLaren,NOR,17.0,NaN
9,2023-03-05 15:00:00+00:00,Aston Martin,ALO,3.0,NaN


In [50]:
# сортируем по пилоту, трассе и дате
race = race.sort_values(["driver_number", "circuit_short_name", "date_start"]).reset_index(drop=True)

# сдвигаем позицию назад внутри пары пилот+трасса (прячем текущую гонку)
race["pos_shifted"] = race.groupby(["driver_number", "circuit_short_name"])["position"].shift(1)

# средняя позиция пилота на ЭТОЙ трассе в прошлом
race["driver_track_history"] = (
    race.groupby(["driver_number", "circuit_short_name"])["pos_shifted"].expanding().mean().values
)

# убираем временную колонку
race = race.drop(columns=["pos_shifted"])

# возвращаем сортировку по дате
race = race.sort_values("date_start").reset_index(drop=True)

race[["date_start", "name_acronym", "circuit_short_name", "position", "driver_track_history"]].head(15)

,date_start,name_acronym,circuit_short_name,position,driver_track_history
0,2023-03-05 15:00:00+00:00,HAM,Sakhir,5.0,NaN
1,2023-03-05 15:00:00+00:00,ZHO,Sakhir,16.0,NaN
2,2023-03-05 15:00:00+00:00,DEV,Sakhir,14.0,NaN
3,2023-03-05 15:00:00+00:00,PER,Sakhir,2.0,NaN
4,2023-03-05 15:00:00+00:00,NOR,Sakhir,17.0,NaN
5,2023-03-05 15:00:00+00:00,STR,Sakhir,6.0,NaN
6,2023-03-05 15:00:00+00:00,VER,Sakhir,1.0,NaN
7,2023-03-05 15:00:00+00:00,HUL,Sakhir,15.0,NaN
8,2023-03-05 15:00:00+00:00,ALO,Sakhir,3.0,NaN
9,2023-03-05 15:00:00+00:00,SAI,Sakhir,4.0,NaN


In [51]:
# отметим, финишировал ли пилот (не сошел)
# dnf = True значит сход, поэтому finished = противоположное
race["finished"] = (race["dnf"] == False).astype(int)

# сортируем по пилоту и дате
race = race.sort_values(["driver_number", "date_start"]).reset_index(drop=True)

# сдвигаем флаг финиша назад (прячем текущую гонку)
race["fin_shifted"] = race.groupby("driver_number")["finished"].shift(1)

# доля прошлых гонок, которые пилот завершил без схода
race["driver_finish_rate"] = (
    race.groupby("driver_number")["fin_shifted"].expanding().mean().values
)

# убираем временные колонки
race = race.drop(columns=["fin_shifted"])

# возвращаем сортировку по дате
race = race.sort_values("date_start").reset_index(drop=True)

print(race[["date_start", "name_acronym", "dnf", "finished", "driver_finish_rate"]].head(15))

                  date_start name_acronym    dnf  finished  driver_finish_rate
0  2023-03-05 15:00:00+00:00          VER  False         1                 NaN
1  2023-03-05 15:00:00+00:00          SAR  False         1                 NaN
2  2023-03-05 15:00:00+00:00          ALB  False         1                 NaN
3  2023-03-05 15:00:00+00:00          OCO   True         0                 NaN
4  2023-03-05 15:00:00+00:00          TSU  False         1                 NaN
5  2023-03-05 15:00:00+00:00          NOR  False         1                 NaN
6  2023-03-05 15:00:00+00:00          DEV  False         1                 NaN
7  2023-03-05 15:00:00+00:00          MAG  False         1                 NaN
8  2023-03-05 15:00:00+00:00          HUL  False         1                 NaN
9  2023-03-05 15:00:00+00:00          BOT  False         1                 NaN
10 2023-03-05 15:00:00+00:00          HAM  False         1                 NaN
11 2023-03-05 15:00:00+00:00          GAS  False    

In [52]:
# сортируем по пилоту и дате
race = race.sort_values(["driver_number", "date_start"]).reset_index(drop=True)

# сколько гонок пилот проехал ДО текущей
# cumcount считает с нуля и не включает текущую строку -> как раз прошлые гонки
race["driver_experience"] = race.groupby("driver_number").cumcount()

# возвращаем сортировку по дате
race = race.sort_values("date_start").reset_index(drop=True)

race[["date_start", "name_acronym", "driver_experience"]].head(15)

,date_start,name_acronym,driver_experience
0,2023-03-05 15:00:00+00:00,VER,0
1,2023-03-05 15:00:00+00:00,SAR,0
2,2023-03-05 15:00:00+00:00,ALB,0
3,2023-03-05 15:00:00+00:00,OCO,0
4,2023-03-05 15:00:00+00:00,TSU,0
5,2023-03-05 15:00:00+00:00,NOR,0
6,2023-03-05 15:00:00+00:00,DEV,0
7,2023-03-05 15:00:00+00:00,MAG,0
8,2023-03-05 15:00:00+00:00,HUL,0
9,2023-03-05 15:00:00+00:00,BOT,0


In [53]:
# сортируем по пилоту и дате
race = race.sort_values(["driver_number", "date_start"]).reset_index(drop=True)

# сдвигаем макс скорость назад (прячем текущую гонку)
race["speed_shifted"] = race.groupby("driver_number")["max_st_speed"].shift(1)

# средняя макс скорость пилота за последние 5 прошлых гонок
race["driver_speed_5"] = (
    race.groupby("driver_number")["speed_shifted"].rolling(5).mean().values
)

# убираем временную колонку
race = race.drop(columns=["speed_shifted"])

# возвращаем сортировку по дате
race = race.sort_values("date_start").reset_index(drop=True)

print(race[["date_start", "name_acronym", "max_st_speed", "driver_speed_5"]].head(15))

                  date_start name_acronym  max_st_speed  driver_speed_5
0  2023-03-05 15:00:00+00:00          VER         303.0             NaN
1  2023-03-05 15:00:00+00:00          SAR         328.0             NaN
2  2023-03-05 15:00:00+00:00          ALB         324.0             NaN
3  2023-03-05 15:00:00+00:00          OCO         329.0             NaN
4  2023-03-05 15:00:00+00:00          TSU         323.0             NaN
5  2023-03-05 15:00:00+00:00          NOR         325.0             NaN
6  2023-03-05 15:00:00+00:00          DEV         320.0             NaN
7  2023-03-05 15:00:00+00:00          MAG         315.0             NaN
8  2023-03-05 15:00:00+00:00          HUL         328.0             NaN
9  2023-03-05 15:00:00+00:00          BOT         320.0             NaN
10 2023-03-05 15:00:00+00:00          HAM         333.0             NaN
11 2023-03-05 15:00:00+00:00          GAS         327.0             NaN
12 2023-03-05 15:00:00+00:00          LEC         324.0         

In [54]:
# список наших новых фич
feature_cols = [
    "driver_form_all",
    "driver_form_5",
    "team_form_5",
    "driver_track_history",
    "driver_finish_rate",
    "driver_experience",
    "driver_speed_5",
]

# сколько пропусков в каждой фиче
print("пропуски (NaN) по фичам:")
print(race[feature_cols].isna().sum())
print()
print("всего строк:", len(race))

пропуски (NaN) по фичам:
driver_form_all          38
driver_form_5           690
team_form_5             615
driver_track_history    702
driver_finish_rate       32
driver_experience         0
driver_speed_5          252
dtype: int64

всего строк: 1397


In [55]:
# создаем позицию для расчета формы, где сходы = последнее место
# для каждой гонки берем максимальную позицию финишировавших и сходам даем хуже

# сколько пилотов в каждой гонке (по session_key)
race["field_size"] = race.groupby("session_key")["driver_number"].transform("count")

# position_filled: если позиция есть, берем ее; если сход (NaN), ставим последнее место (field_size)
race["position_filled"] = race["position"].fillna(race["field_size"])

# проверяем
print(race[["name_acronym", "position", "dnf", "field_size", "position_filled"]].head(15))

   name_acronym  position    dnf  field_size  position_filled
0           VER       1.0  False          20              1.0
1           SAR      12.0  False          20             12.0
2           ALB      10.0  False          20             10.0
3           OCO       NaN   True          20             20.0
4           TSU      11.0  False          20             11.0
5           NOR      17.0  False          20             17.0
6           DEV      14.0  False          20             14.0
7           MAG      13.0  False          20             13.0
8           HUL      15.0  False          20             15.0
9           BOT       8.0  False          20              8.0
10          HAM       5.0  False          20              5.0
11          GAS       9.0  False          20              9.0
12          LEC       NaN   True          20             20.0
13          RUS       7.0  False          20              7.0
14          PER       2.0  False          20              2.0


In [56]:
# пересчитываем форму пилота, используя position_filled (сходы учтены как последнее место)
race = race.sort_values(["driver_number", "date_start"]).reset_index(drop=True)
race["pos_shifted"] = race.groupby("driver_number")["position_filled"].shift(1)

# форма за все прошлые гонки
race["driver_form_all"] = race.groupby("driver_number")["pos_shifted"].expanding().mean().values

# форма за последние 5 гонок (min_periods=1: считаем даже если гонок меньше 5)
race["driver_form_5"] = (
    race.groupby("driver_number")["pos_shifted"]
    .rolling(5, min_periods=1).mean().values
)
race = race.drop(columns=["pos_shifted"])

# форма команды за последние 5 гонок
race = race.sort_values(["team_name", "date_start"]).reset_index(drop=True)
race["pos_shifted"] = race.groupby("team_name")["position_filled"].shift(1)
race["team_form_5"] = (
    race.groupby("team_name")["pos_shifted"]
    .rolling(5, min_periods=1).mean().values
)
race = race.drop(columns=["pos_shifted"])

# история на трассе
race = race.sort_values(["driver_number", "circuit_short_name", "date_start"]).reset_index(drop=True)
race["pos_shifted"] = race.groupby(["driver_number", "circuit_short_name"])["position_filled"].shift(1)
race["driver_track_history"] = (
    race.groupby(["driver_number", "circuit_short_name"])["pos_shifted"]
    .expanding().mean().values
)
race = race.drop(columns=["pos_shifted"])

# возвращаем сортировку по дате
race = race.sort_values("date_start").reset_index(drop=True)

# смотрим, сколько пропусков осталось
print("пропуски после пересчета:")
print(race[feature_cols].isna().sum())

пропуски после пересчета:
driver_form_all          32
driver_form_5            32
team_form_5              13
driver_track_history    642
driver_finish_rate       32
driver_experience         0
driver_speed_5          252
dtype: int64


In [57]:
# флаг дебютанта: мало опыта (меньше 5 гонок в датасете)
race["is_rookie"] = (race["driver_experience"] < 5).astype(int)

# фичи-позиции заполняем медианой (середина пелотона)
position_features = ["driver_form_all", "driver_form_5", "team_form_5", "driver_track_history"]
for col in position_features:
    race[col] = race[col].fillna(race[col].median())

# надежность заполняем медианой
race["driver_finish_rate"] = race["driver_finish_rate"].fillna(race["driver_finish_rate"].median())

# скорость заполняем медианой
race["driver_speed_5"] = race["driver_speed_5"].fillna(race["driver_speed_5"].median())

# проверяем, что пропусков не осталось
print("пропуски после заполнения:")
print(race[feature_cols].isna().sum())
print()
print("дебютантов (is_rookie=1):", race["is_rookie"].sum(), "из", len(race))

пропуски после заполнения:
driver_form_all         0
driver_form_5           0
team_form_5             0
driver_track_history    0
driver_finish_rate      0
driver_experience       0
driver_speed_5          0
dtype: int64

дебютантов (is_rookie=1): 149 из 1397


In [60]:
# одна и та же команда под разными именами в разные годы -> приводим к единому виду
team_mapping = {
    "AlphaTauri": "Racing Bulls",
    "RB": "Racing Bulls",             # AlphaTauri переименовали в RB (2024)
    "Alfa Romeo": "Kick Sauber",     # Alfa Romeo стали Kick Sauber (2024)
}

# применяем маппинг: если имя есть в словаре, меняем, иначе оставляем как было
race["team_name"] = race["team_name"].replace(team_mapping)

# смотрим, какие команды остались
print("команды после объединения:")
race["team_name"].value_counts()

команды после объединения:


team_name
Mercedes           140
Kick Sauber        140
Racing Bulls       140
Red Bull Racing    140
McLaren            140
Haas F1 Team       140
Ferrari            140
Alpine             140
Williams           139
Aston Martin       138
Name: count, dtype: int64

In [62]:
# пересчитываем форму команды, теперь имена объединены
race = race.sort_values(["team_name", "date_start"]).reset_index(drop=True)
race["pos_shifted"] = race.groupby("team_name")["position_filled"].shift(1)

race["team_form_5"] = (
    race.groupby("team_name")["pos_shifted"]
    .rolling(5, min_periods=1).mean().values
)
race = race.drop(columns=["pos_shifted"])

# заполняем пропуски (первые гонки) медианой
race["team_form_5"] = race["team_form_5"].fillna(race["team_form_5"].median())

# возвращаем сортировку по дате
race = race.sort_values("date_start").reset_index(drop=True)

print("пропуски team_form_5:", race["team_form_5"].isna().sum())
race[["date_start", "team_name", "name_acronym", "position", "team_form_5"]]

пропуски team_form_5: 0


,date_start,team_name,name_acronym,position,team_form_5
0,2023-03-05 15:00:00+00:00,Alpine,GAS,9.0,11.4
1,2023-03-05 15:00:00+00:00,Ferrari,SAI,4.0,11.4
2,2023-03-05 15:00:00+00:00,Ferrari,LEC,NaN,4.0
3,2023-03-05 15:00:00+00:00,Haas F1 Team,HUL,15.0,11.4
4,2023-03-05 15:00:00+00:00,Haas F1 Team,MAG,13.0,15.0
...,...,...,...,...,...
1392,2025-12-07 13:00:00+00:00,Alpine,GAS,19.0,15.6
1393,2025-12-07 13:00:00+00:00,Ferrari,HAM,8.0,7.2
1394,2025-12-07 13:00:00+00:00,Ferrari,LEC,4.0,10.4
1395,2025-12-07 13:00:00+00:00,Aston Martin,ALO,6.0,13.8


In [67]:
# берем позицию из квалы (где пилот квалифицировался)
# у квалы и гонки одного уикенда общий meeting_key + driver_number
quali_pos = quali_results[["meeting_key", "driver_number", "position"]].copy()
quali_pos = quali_pos.rename(columns={"position": "quali_position"})

# приклеиваем к гонке по уикенду и пилоту
race = race.merge(quali_pos, on=["meeting_key", "driver_number"], how="left")

print("пропусков в quali_position:", race["quali_position"].isna().sum())
race[["date_start", "name_acronym", "quali_position", "position"]].head(15)

пропусков в quali_position: 42


,date_start,name_acronym,quali_position,position
0,2023-03-05 15:00:00+00:00,GAS,20.0,9.0
1,2023-03-05 15:00:00+00:00,SAI,4.0,4.0
2,2023-03-05 15:00:00+00:00,LEC,3.0,NaN
3,2023-03-05 15:00:00+00:00,HUL,10.0,15.0
4,2023-03-05 15:00:00+00:00,MAG,17.0,13.0
5,2023-03-05 15:00:00+00:00,ZHO,13.0,16.0
6,2023-03-05 15:00:00+00:00,BOT,12.0,8.0
7,2023-03-05 15:00:00+00:00,NOR,11.0,17.0
8,2023-03-05 15:00:00+00:00,HAM,7.0,5.0
9,2023-03-05 15:00:00+00:00,RUS,6.0,7.0


In [ ]:
# пропуски в quali_position заполняем последним местом
# логика: если квалы нет, считаем что стартовал сзади (консервативно)
race["quali_position"] = race["quali_position"].fillna(race["quali_position"].max())

print("пропусков в quali_position:", race["quali_position"].isna().sum())
print("диапазон:", race["quali_position"].min(), "-", race["quali_position"].max())

In [69]:
import os
os.makedirs("../data/data_final", exist_ok=True)

race.to_csv("../data/data_final/race_features.csv", index=False)

print("сохранено:", race.shape)

сохранено: (1397, 45)
